In [1]:
import pandas as pd
import numpy as np

In [2]:
# 1. Đọc file dữ liệu chính
df = pd.read_csv("order_items.csv", low_memory=False)

# 2. Đọc các bảng tham chiếu để kiểm tra khóa ngoại
df_orders = pd.read_csv("orders_enriched.csv", low_memory=False)
df_products = pd.read_csv("products.csv", low_memory=False)
df_promotions = pd.read_csv("promotions.csv", low_memory=False)

print(f"Kích thước order_items ban đầu: {df.shape}")

Kích thước order_items ban đầu: (714669, 7)


In [3]:
print("5 DÒNG DỮ LIỆU ĐẦU TIÊN:")
display(df.head())

print("\nTHÔNG TIN CÁC CỘT & KIỂU DỮ LIỆU:")
df.info()

5 DÒNG DỮ LIỆU ĐẦU TIÊN:


,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
0,1,2400,7,1138.22,0.0,NaN,NaN
1,2,609,7,10166.25,0.0,NaN,NaN
2,3,396,3,11220.33,0.0,NaN,NaN
3,4,635,5,10639.25,0.0,NaN,NaN
4,6,1935,1,1597.84,0.0,NaN,NaN



THÔNG TIN CÁC CỘT & KIỂU DỮ LIỆU:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order_id         714669 non-null  int64  
 1   product_id       714669 non-null  int64  
 2   quantity         714669 non-null  int64  
 3   unit_price       714669 non-null  float64
 4   discount_amount  714669 non-null  float64
 5   promo_id         276316 non-null  object 
 6   promo_id_2       206 non-null     object 
dtypes: float64(2), int64(3), object(2)
memory usage: 38.2+ MB


In [4]:
pk_cols = ["order_id", "product_id"]
print("2. KIỂM TRA TRÙNG LẶP:")
print("Số dòng trùng 100% tất cả các cột:", df.duplicated().sum())
print(f"Số dòng trùng khóa chính ({', '.join(pk_cols)}):", df.duplicated(subset=pk_cols).sum())

2. KIỂM TRA TRÙNG LẶP:
Số dòng trùng 100% tất cả các cột: 0
Số dòng trùng khóa chính (order_id, product_id): 16


In [5]:
print("3. KIỂM TRA LOGIC NGHIỆP VỤ ORDER_ITEMS:")
print("Số dòng có số lượng (quantity) <= 0           :", (df["quantity"] <= 0).sum())
print("Số dòng có đơn giá (unit_price) < 0           :", (df["unit_price"] < 0).sum())
print("Số dòng có tiền giảm giá (discount_amount) < 0:", (df["discount_amount"] < 0).sum())

# Kiểm tra tiền giảm giá có lớn hơn tổng giá trị hàng (quantity * unit_price) không
total_item_val = df["quantity"] * df["unit_price"]
print("Số dòng có discount_amount > Tổng tiền hàng   :", (df["discount_amount"] > total_item_val).sum())

3. KIỂM TRA LOGIC NGHIỆP VỤ ORDER_ITEMS:
Số dòng có số lượng (quantity) <= 0           : 0
Số dòng có đơn giá (unit_price) < 0           : 0
Số dòng có tiền giảm giá (discount_amount) < 0: 0
Số dòng có discount_amount > Tổng tiền hàng   : 0


In [6]:
print("4. KIỂM TRA RÀNG BUỘC KHÓA NGOẠI (REFERENTIAL INTEGRITY):")

# Tạo tập hợp các ID hợp lệ
valid_order_ids = set(df_orders['order_id'].dropna().astype(str).str.strip())
valid_product_ids = set(df_products['product_id'].dropna().astype(str).str.strip())
valid_promo_ids = set(df_promotions['promo_id'].dropna().astype(str).str.strip()) if 'promo_id' in df_promotions.columns else set()

# Báo cáo các dòng vi phạm
invalid_orders = df[~df['order_id'].astype(str).str.strip().isin(valid_order_ids)]
invalid_products = df[~df['product_id'].astype(str).str.strip().isin(valid_product_ids)]

print(f"Số dòng order_id KHÔNG tồn tại trong bảng Orders     : {len(invalid_orders)}")
print(f"Số dòng product_id KHÔNG tồn tại trong bảng Products : {len(invalid_products)}")

if 'promo_id' in df.columns and valid_promo_ids:
    has_promo = df[df['promo_id'].notnull() & ~df['promo_id'].astype(str).isin(['nan', 'NaN', 'None'])]
    invalid_promos = has_promo[~has_promo['promo_id'].astype(str).str.strip().isin(valid_promo_ids)]
    print(f"Số dòng promo_id KHÔNG tồn tại trong bảng Promotions : {len(invalid_promos)}")

4. KIỂM TRA RÀNG BUỘC KHÓA NGOẠI (REFERENTIAL INTEGRITY):
Số dòng order_id KHÔNG tồn tại trong bảng Orders     : 0
Số dòng product_id KHÔNG tồn tại trong bảng Products : 0
Số dòng promo_id KHÔNG tồn tại trong bảng Promotions : 0


In [7]:
# 1. Lọc dữ liệu hợp lệ theo logic nghiệp vụ
df_clean = df[(df['quantity'] > 0) & (df['unit_price'] >= 0) & (df['discount_amount'] >= 0)].copy()

# 2. Lọc chỉ giữ các dòng có order_id và product_id hợp lệ từ các bảng danh mục
df_clean = df_clean[
    df_clean['order_id'].astype(str).str.strip().isin(valid_order_ids) &
    df_clean['product_id'].astype(str).str.strip().isin(valid_product_ids)
]

print(f"Kích thước bảng order_items sau khi lọc làm sạch: {df_clean.shape}")

Kích thước bảng order_items sau khi lọc làm sạch: (714669, 7)


In [8]:
# 1. Ép kiểu dữ liệu số nguyên
df_clean["order_id"] = df_clean["order_id"].astype("int32")
df_clean["product_id"] = df_clean["product_id"].astype("int32")
df_clean["quantity"] = df_clean["quantity"].astype("int32")

# 2. Làm sạch chuỗi "nan" ở các cột mã khuyến mãi
if "promo_id" in df_clean.columns:
    df_clean["promo_id"] = df_clean["promo_id"].astype(str).str.strip().replace(["nan", "NaN", "NAN", "None"], None)
if "promo_id_2" in df_clean.columns:
    df_clean["promo_id_2"] = df_clean["promo_id_2"].astype(str).str.strip().replace(["nan", "NaN", "NAN", "None"], None)

print("Đã hoàn tất ép kiểu dữ liệu và làm sạch chuỗi.")

Đã hoàn tất ép kiểu dữ liệu và làm sạch chuỗi.


In [9]:
# Xác định danh sách cột nhóm
group_cols = ["order_id", "product_id"]
if "promo_id" in df_clean.columns: group_cols.append("promo_id")
if "promo_id_2" in df_clean.columns: group_cols.append("promo_id_2")

# Gộp các dòng bị lặp khóa chính: cộng dồn quantity/discount, lấy giá trung bình unit_price
df_clean = df_clean.groupby(group_cols, as_index=False, dropna=False).agg({
    "quantity": "sum",
    "unit_price": "mean",
    "discount_amount": "sum"
})

# Đảm bảo thứ tự các cột khớp 100% với ERD
erd_columns = [col for col in ["order_id", "product_id", "promo_id", "promo_id_2", "quantity", "unit_price", "discount_amount"] if col in df_clean.columns]
df_clean = df_clean[erd_columns]

# Sắp xếp lại theo order_id và product_id
df_clean = df_clean.sort_values(["order_id", "product_id"]).reset_index(drop=True)

print("Đã hoàn tất xử lý gộp trùng và sắp xếp thứ tự cột.")

Đã hoàn tất xử lý gộp trùng và sắp xếp thứ tự cột.


In [13]:
output_path = "Order_items.csv"
df_clean.to_csv("Order_item.csv", index=False,sep=',', encoding="utf-8-sig")

print("=== THÔNG TIN BẢNG ORDER_ITEMS SAU KHI LÀM SẠCH ===")
print(f"Tong so dong: {len(df_clean)}")
print(f"Tong so cot : {len(df_clean.columns)}")
print(f"Da xuat file thanh cong tai: {output_path}")

=== THÔNG TIN BẢNG ORDER_ITEMS SAU KHI LÀM SẠCH ===
Tong so dong: 714653
Tong so cot : 7
Da xuat file thanh cong tai: Order_items.csv
